# LeetCode 491 — Non-decreasing Subsequences bằng Python

Link đề: https://leetcode.com/problems/non-decreasing-subsequences/ · Độ khó: **Medium** · Chủ đề: Backtracking, Array, Hash Table, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc. Thứ tự các dãy trong kết quả **không quan trọng** (đề cho phép trả về theo thứ tự bất kỳ), nên khi kiểm tra trong notebook ta luôn so sánh sau khi sắp xếp. Mọi đáp án được đối chiếu với một oracle vét cạn độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: liệt kê **tất cả dãy con (subsequence) không giảm, khác nhau, có ít nhất 2 phần tử** của mảng `nums`.
2. Nắm công cụ Python: `set` để khử trùng lặp, `tuple` làm khóa của set (vì `list` không băm được), phép dịch bit `1 << n`, `itertools.combinations`, `list.append` / `list.pop` cho backtracking.
3. Đi từ **vét cạn mọi tập con bằng bitmask** → **backtracking + set khử trùng cuối cùng** → **backtracking khử trùng ngay trong từng tầng đệ quy** → **quyết định "chọn / bỏ" có luật chống trùng** → **xây dần tập dãy theo từng phần tử (DP kiểu tập)**.
4. Hiểu mấu chốt: trùng lặp sinh ra khi **cùng một giá trị được chọn ở cùng một "vị trí trong dãy" nhưng từ hai chỉ số khác nhau**; chỉ cần mỗi tầng đệ quy không dùng cùng một giá trị hai lần là hết trùng — **không cần sắp xếp mảng** (và cũng không được sắp xếp, vì sẽ phá thứ tự gốc).
5. Tránh các lỗi kinh điển: dùng `used` toàn cục thay vì theo tầng, quên `copy` của path (`ans.append(path)` thay vì `path[:]`), sắp xếp mảng đầu vào, thêm cả dãy dài 1, quên điều kiện `>=` (cho phép bằng nhau).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Hãy trả về **mọi dãy con không giảm khác nhau** của `nums` có **ít nhất hai phần tử**. Bạn có thể trả về đáp án theo **thứ tự bất kỳ**.

- **Dãy con (subsequence)** là dãy nhận được bằng cách xóa một số phần tử (có thể không xóa) nhưng **giữ nguyên thứ tự** các phần tử còn lại. Không cần liên tiếp.
- **Không giảm (non-decreasing)**: mỗi phần tử `>=` phần tử đứng trước nó (cho phép bằng nhau).
- **Khác nhau**: hai dãy có cùng giá trị từng vị trí chỉ được tính một lần, dù chúng được tạo từ các chỉ số khác nhau.

**Ví dụ 1:** `nums = [4,6,7,7]` → `[[4,6],[4,6,7],[4,6,7,7],[4,7],[4,7,7],[6,7],[6,7,7],[7,7]]`

**Ví dụ 2:** `nums = [4,4,3,2,1]` → `[[4,4]]`

**Ràng buộc:**

- `1 <= nums.length <= 15`
- `-100 <= nums[i] <= 100`

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [
    ([4, 6, 7, 7], [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]),
    ([4, 4, 3, 2, 1], [[4, 4]]),
]
for nums, expected in examples:
    print(nums, "->", len(expected), "dãy:", expected)

[4, 6, 7, 7] -> 8 dãy: [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
[4, 4, 3, 2, 1] -> 1 dãy: [[4, 4]]


### 2.1. Đọc kỹ đề

- `n <= 15` rất nhỏ: có tối đa `2^15 = 32768` dãy con, nên **vét cạn được phép** — đây là gợi ý lớn rằng đề muốn backtracking / bitmask.
- Với `[4,6,7,7]` có **hai** số `7` ở chỉ số 2 và 3. Dãy `[4,7]` có thể tạo bằng `(chỉ số 0, chỉ số 2)` hoặc `(chỉ số 0, chỉ số 3)` — nhưng đề chỉ muốn **một** `[4,7]`. Đây là chỗ "khác nhau" gây khó.
- Không được sắp xếp `nums` trước: với `nums = [3,1,2]`, `[1,3]` **không** phải dãy con (trong mảng gốc `3` đứng trước `1`), nhưng sau khi sắp xếp thành `[1,2,3]` thì nó lại xuất hiện. Dãy con phụ thuộc **thứ tự gốc**.
- Dãy chỉ có **1** phần tử không được tính (cần ít nhất 2).

## 3. Kiến thức nền

### 3.1. Dãy con và số lượng dãy con

Mảng `n` phần tử có `2^n` tập con theo chỉ số (mỗi phần tử: chọn hoặc không chọn). Với `n = 4` là 16. Ta liệt kê thử bằng `itertools.combinations(chỉ_số, k)`.

In [2]:
nums = [4, 6, 7, 7]
idx = range(len(nums))
total = 0
for k in range(len(nums) + 1):
    for pick in combinations(idx, k):
        total += 1
        if k == 2:
            print("chọn chỉ số", pick, "->", [nums[i] for i in pick])
print("tổng số tập con:", total, "= 2 **", len(nums), "=", 2 ** len(nums))

chọn chỉ số (0, 1) -> [4, 6]
chọn chỉ số (0, 2) -> [4, 7]
chọn chỉ số (0, 3) -> [4, 7]
chọn chỉ số (1, 2) -> [6, 7]
chọn chỉ số (1, 3) -> [6, 7]
chọn chỉ số (2, 3) -> [7, 7]
tổng số tập con: 16 = 2 ** 4 = 16


### 3.2. Bitmask: dùng một số nguyên để biểu diễn tập con

Mỗi số `mask` từ `0` đến `2^n - 1` là một tập con: **bit thứ `i` bằng 1 nghĩa là chọn `nums[i]`**. Kiểm tra bit: `(mask >> i) & 1`. `1 << n` chính là `2^n`.

In [3]:
nums = [4, 6, 7, 7]
n = len(nums)
print("1 << n =", 1 << n)
for mask in [0b0000, 0b0011, 0b0101, 0b1011, 0b1111]:
    chosen = [nums[i] for i in range(n) if (mask >> i) & 1]
    print(f"mask={mask:04b} ({mask:>2}) -> {chosen}")

1 << n = 16
mask=0000 ( 0) -> []
mask=0011 ( 3) -> [4, 6]
mask=0101 ( 5) -> [4, 7]
mask=1011 (11) -> [4, 6, 7]
mask=1111 (15) -> [4, 6, 7, 7]


### 3.3. `set` và `tuple`: khử trùng lặp

`set` giữ mỗi phần tử một lần. Nhưng `list` **không** băm được nên không thể bỏ vào `set`; ta đổi sang `tuple` (bất biến, băm được) rồi khi cần mới đổi lại thành `list`.

In [4]:
seen = set()
seen.add((4, 7))
seen.add((4, 7))          # trùng -> bị bỏ qua
seen.add((7, 7))
print(seen, len(seen))
print((4, 7) in seen, (6, 7) in seen)
try:
    seen.add([4, 7])      # list không băm được
except TypeError as e:
    print("Lỗi:", e)
print([list(t) for t in sorted(seen)])

{(4, 7), (7, 7)} 2
True False
Lỗi: unhashable type: 'list'
[[4, 7], [7, 7]]


### 3.4. Backtracking và quy tắc `append` / `pop`

Backtracking = thử một lựa chọn, đệ quy vào sâu, rồi **hoàn tác** lựa chọn để thử cái khác. Với danh sách `path`: `path.append(x)` rồi `path.pop()`. Khi chốt đáp án phải lưu **bản sao** `path[:]` (hoặc `list(path)`), vì `path` còn tiếp tục bị sửa.

In [5]:
path = []
ans = []
path.append(1); path.append(2)
ans.append(path)          # SAI: lưu tham chiếu
ans2 = [path[:]]          # ĐÚNG: lưu bản sao
path.pop()
print("ans  =", ans, " <- bị đổi theo path!")
print("ans2 =", ans2, " <- giữ nguyên")

ans  = [[1]]  <- bị đổi theo path!
ans2 = [[1, 2]]  <- giữ nguyên


### 3.5. Tên tham số mặc định và hàm lồng

Trong LeetCode ta thường viết hàm đệ quy `dfs` **bên trong** method để nó dùng được `nums`, `ans`, `path` mà không cần truyền. Từ khóa `nonlocal` chỉ cần khi **gán lại** biến ngoài; với `list.append` thì không cần.

In [6]:
def demo():
    ans = []
    def dfs(i):
        if i == 3:
            return
        ans.append(i)          # sửa nội dung list ngoài: không cần nonlocal
        dfs(i + 1)
    dfs(0)
    return ans
print(demo())

[0, 1, 2]


## 4. Ý tưởng nền tảng: tính tay

Lấy `nums = [4, 6, 7, 7]`. Ta liệt kê tay các dãy con không giảm dài `>= 2` theo **chỉ số**:

- Bắt đầu từ `4` (chỉ số 0): `4,6` · `4,7(2)` · `4,7(3)` · `4,6,7(2)` · `4,6,7(3)` · `4,7(2),7(3)` · `4,6,7(2),7(3)`
- Bắt đầu từ `6` (chỉ số 1): `6,7(2)` · `6,7(3)` · `6,7(2),7(3)`
- Bắt đầu từ `7` (chỉ số 2): `7(2),7(3)`

Có `7 + 3 + 1 = 11` dãy theo chỉ số, nhưng **theo giá trị** chỉ còn `8` vì: `4,7` xuất hiện 2 lần, `4,6,7` 2 lần, `6,7` 2 lần (ba cặp trùng ⇒ 11 − 3 = 8).

> Trực giác: **mỗi dãy theo giá trị** chỉ nên được sinh ra **một lần**. Hai đường đi chỉ khác nhau ở việc "lấy số `7` thứ nhất hay thứ hai" phải bị gộp lại.

In [7]:
nums = [4, 6, 7, 7]
by_index = []
for k in range(2, len(nums) + 1):
    for pick in combinations(range(len(nums)), k):
        vals = [nums[i] for i in pick]
        if all(vals[j] <= vals[j + 1] for j in range(len(vals) - 1)):
            by_index.append((pick, vals))
print("theo chỉ số:", len(by_index))
for pick, vals in by_index:
    print(pick, vals)
print("theo giá trị:", len({tuple(v) for _, v in by_index}))

theo chỉ số: 11
(0, 1) [4, 6]
(0, 2) [4, 7]
(0, 3) [4, 7]
(1, 2) [6, 7]
(1, 3) [6, 7]
(2, 3) [7, 7]
(0, 1, 2) [4, 6, 7]
(0, 1, 3) [4, 6, 7]
(0, 2, 3) [4, 7, 7]
(1, 2, 3) [6, 7, 7]
(0, 1, 2, 3) [4, 6, 7, 7]
theo giá trị: 8


## 5. Cách 1 — Brute Force: thử mọi tập con bằng bitmask

**Ý tưởng.** Duyệt `mask` từ `1` đến `2^n - 1`. Với mỗi `mask`, dựng dãy con theo các bit 1, kiểm tra:

1. có ít nhất 2 phần tử;
2. không giảm (mỗi cặp liền kề `a <= b`).

Nếu thỏa, thêm `tuple(dãy)` vào một `set` để loại trùng. Cuối cùng đổi ra list.

**Code demo:**

In [8]:
def find_brute(nums):
    n = len(nums)
    found = set()
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if (mask >> i) & 1]
        if len(seq) < 2:
            continue
        if all(seq[j] <= seq[j + 1] for j in range(len(seq) - 1)):
            found.add(tuple(seq))
    return [list(t) for t in found]


for nums, expected in examples:
    got = find_brute(nums)
    print(nums, "->", sorted(got))
    assert sorted(got) == sorted(expected)

[4, 6, 7, 7] -> [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
[4, 4, 3, 2, 1] -> [[4, 4]]


**Code nộp LeetCode (Cách 1):**

In [9]:
class SolutionBrute:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        n = len(nums)
        found = set()
        for mask in range(1, 1 << n):
            seq = [nums[i] for i in range(n) if (mask >> i) & 1]
            if len(seq) >= 2 and all(seq[j] <= seq[j + 1] for j in range(len(seq) - 1)):
                found.add(tuple(seq))
        return [list(t) for t in found]


for nums, expected in examples:
    assert sorted(SolutionBrute().findSubsequences(nums)) == sorted(expected)
print("Cách 1 OK")

Cách 1 OK


**Độ phức tạp.** Có `2^n` mask, mỗi mask tốn `O(n)` để dựng và kiểm tra ⇒ **`O(n · 2^n)`** thời gian; set có thể chứa tới `O(2^n)` dãy ⇒ `O(n · 2^n)` bộ nhớ trong trường hợp xấu.

**Nút thắt.** Ta dựng **cả những tập con chắc chắn hỏng** (ví dụ `[7, 4, ...]`) rồi mới kiểm tra. Backtracking sẽ **cắt sớm**: ngay khi thêm một số nhỏ hơn số cuối thì bỏ nhánh đó, không dựng tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Sinh dãy dần dần.** Ta xây dãy theo thứ tự chỉ số tăng. Tại mỗi chỉ số `i` có hai quyết định: **lấy** `nums[i]` (nếu `>=` số cuối của dãy hiện tại) hoặc **bỏ**. Mỗi khi dãy có `>= 2` phần tử, ta ghi nhận nó.

**Bước 2 — Vì sao bị trùng?** Với `[4,6,7,7]`: đường đi "lấy 4, bỏ 6, lấy 7(chỉ số 2), bỏ 7(chỉ số 3)" và "lấy 4, bỏ 6, bỏ 7(chỉ số 2), lấy 7(chỉ số 3)" cho cùng `[4,7]`.

**Bước 3 — Chặn trùng.** Có hai cách chặn đẹp:

- **Cách A (theo tầng):** ở mỗi lần gọi đệ quy, ta chọn **phần tử kế tiếp** của dãy trong `nums[start:]`. Nếu cùng giá trị đã được chọn làm "phần tử kế tiếp" trong lần gọi này rồi, **bỏ qua** (dùng một `set` cục bộ cho lần gọi).
- **Cách B (chọn/bỏ có luật):** khi **không** lấy `nums[u]`, chỉ được bỏ nếu `nums[u] != số cuối`. Nếu `nums[u] == số cuối`, ta **bắt buộc lấy** (vì bỏ rồi lấy bản sao sau sẽ trùng).

**Bước 4 — Hoặc xây theo tập.** Ngược lại với đệ quy, duyệt từng phần tử `x`, giữ một `set` các dãy tạo được; với mỗi dãy kết thúc `<= x` ta nối `x` vào để được dãy mới.

Mỗi hướng được dạy chi tiết ở mục 7.

## 7. Các cách tiếp theo

### Cách 2 — Backtracking, khử trùng bằng `set` ở cuối

**Giải thích từng đoạn.**

1. `dfs(start, path)`: `path` là dãy đang xây; ta thử mọi chỉ số `i` từ `start` đến `n-1` làm **phần tử tiếp theo**.
2. Chỉ thêm `nums[i]` nếu `path` rỗng **hoặc** `nums[i] >= path[-1]` (cắt nhánh hỏng sớm).
3. Sau khi thêm, nếu `len(path) >= 2` thì lưu `tuple(path)` vào `found` (set tự lo trùng).
4. Đệ quy `dfs(i + 1, path)`, rồi `path.pop()` để hoàn tác.

Cách này **vẫn sinh các đường trùng** (rồi set gom lại) nhưng đã cắt nhánh không hợp lệ, nên nhanh hơn Cách 1.

In [10]:
def find_bt_set(nums):
    n = len(nums)
    found = set()
    path = []

    def dfs(start):
        for i in range(start, n):
            if path and nums[i] < path[-1]:
                continue
            path.append(nums[i])
            if len(path) >= 2:
                found.add(tuple(path))
            dfs(i + 1)
            path.pop()

    dfs(0)
    return [list(t) for t in found]


for nums, expected in examples:
    got = find_bt_set(nums)
    assert sorted(got) == sorted(expected)
    print(nums, "->", sorted(got))

[4, 6, 7, 7] -> [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
[4, 4, 3, 2, 1] -> [[4, 4]]


In [11]:
class SolutionBTSet:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        n = len(nums)
        found = set()
        path = []

        def dfs(start: int) -> None:
            for i in range(start, n):
                if path and nums[i] < path[-1]:
                    continue
                path.append(nums[i])
                if len(path) >= 2:
                    found.add(tuple(path))
                dfs(i + 1)
                path.pop()

        dfs(0)
        return [list(t) for t in found]


for nums, expected in examples:
    assert sorted(SolutionBTSet().findSubsequences(nums)) == sorted(expected)
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp.** Số nút của cây đệ quy tối đa `O(2^n)`, mỗi nút thêm `tuple(path)` tốn `O(n)` ⇒ **`O(n · 2^n)`** thời gian trường hợp xấu; bộ nhớ `O(n · 2^n)` cho set kết quả (tối đa) + `O(n)` ngăn xếp đệ quy. Nhược điểm: với mảng nhiều số trùng (ví dụ toàn `7`) ta lặp lại rất nhiều nhánh cho cùng một dãy rồi mới gom.

### Cách 3 — Backtracking, khử trùng ngay trong từng tầng (chốt nộp bài)

**Giải thích từng đoạn.**

1. Trong mỗi lần gọi `dfs(start)`, tạo `used = set()` **mới**: tập các giá trị đã được thử làm "phần tử kế tiếp" ở lần gọi này.
2. Với `i` từ `start`: nếu `nums[i] < path[-1]` (dãy sẽ giảm) **hoặc** `nums[i] in used` ⇒ bỏ qua.
3. Ngược lại: `used.add(nums[i])`, `path.append`, nếu `len(path) >= 2` thì **ghi nhận ngay vào list kết quả** (không cần set toàn cục vì đảm bảo không trùng), `dfs(i + 1)`, `path.pop()`.

**Vì sao `used` phải theo từng lần gọi, không phải toàn cục?** Hai lần gọi khác nhau ứng với hai **tiền tố khác nhau**; cùng giá trị `7` được phép xuất hiện sau tiền tố `[4]` và cũng sau tiền tố `[6]`. Chỉ trong **cùng một tiền tố** thì mới cấm chọn hai lần cùng giá trị làm phần tử kế tiếp.

In [12]:
def find_bt_level(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(start):
        used = set()                          # theo từng tầng
        for i in range(start, n):
            if (path and nums[i] < path[-1]) or nums[i] in used:
                continue
            used.add(nums[i])
            path.append(nums[i])
            if len(path) >= 2:
                ans.append(path[:])
            dfs(i + 1)
            path.pop()

    dfs(0)
    return ans


for nums, expected in examples:
    got = find_bt_level(nums)
    assert sorted(got) == sorted(expected)
    print(nums, "->", sorted(got))

[4, 6, 7, 7] -> [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
[4, 4, 3, 2, 1] -> [[4, 4]]


**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionBTLevel:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        n = len(nums)
        ans: List[List[int]] = []
        path: List[int] = []

        def dfs(start: int) -> None:
            used = set()
            for i in range(start, n):
                if (path and nums[i] < path[-1]) or nums[i] in used:
                    continue
                used.add(nums[i])
                path.append(nums[i])
                if len(path) >= 2:
                    ans.append(path[:])
                dfs(i + 1)
                path.pop()

        dfs(0)
        return ans


for nums, expected in examples:
    assert sorted(SolutionBTLevel().findSubsequences(nums)) == sorted(expected)
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp.** Mỗi dãy hợp lệ xuất hiện đúng một lần, cây đệ quy có số nút bằng số dãy (kể cả dãy dài 1 là nút trung gian) ⇒ **`O(n · 2^n)`** thời gian trong trường hợp xấu (mảng tăng chặt: gần `2^n` dãy, mỗi dãy sao chép `O(n)`); bộ nhớ phụ ngoài đáp án là `O(n)` (đường đi + `used` mỗi tầng tối đa `n` phần tử, nên `O(n^2)` cộng dồn trong ngăn xếp, vẫn rất nhỏ với `n <= 15`).

### Cách 4 — Quyết định "chọn / bỏ" với luật chống trùng

**Giải thích từng đoạn.**

1. `dfs(u, last)`: đang xét chỉ số `u`, số cuối đã chọn là `last` (ban đầu `-1000`, nhỏ hơn mọi giá trị hợp lệ).
2. Khi `u == n`: nếu `len(path) >= 2` thì ghi nhận, rồi dừng.
3. **Lấy** `nums[u]` nếu `nums[u] >= last`: `path.append`, `dfs(u + 1, nums[u])`, `path.pop()`.
4. **Bỏ** `nums[u]` — chỉ được bỏ khi `nums[u] != last`. Lý do: nếu `nums[u] == last` mà ta bỏ, thì dãy sau đó có thể chọn một số bằng `last` ở chỉ số sau, và dãy nhận được **trùng** với việc "lấy `nums[u]` rồi bỏ số sau". Ép phải lấy ⇒ mỗi dãy chỉ được sinh bởi đúng một đường.

Đây là cách trong lời giải mẫu của doocs, rất gọn: chỉ hai nhánh, không cần `set`.

In [14]:
def find_take_skip(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(u, last):
        if u == n:
            if len(path) >= 2:
                ans.append(path[:])
            return
        if nums[u] >= last:               # nhánh lấy
            path.append(nums[u])
            dfs(u + 1, nums[u])
            path.pop()
        if nums[u] != last:               # nhánh bỏ (chỉ khi không trùng số cuối)
            dfs(u + 1, last)

    dfs(0, -1000)
    return ans


for nums, expected in examples:
    got = find_take_skip(nums)
    assert sorted(got) == sorted(expected)
    print(nums, "->", sorted(got))

[4, 6, 7, 7] -> [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
[4, 4, 3, 2, 1] -> [[4, 4]]


In [15]:
class SolutionTakeSkip:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        n = len(nums)
        ans: List[List[int]] = []
        path: List[int] = []

        def dfs(u: int, last: int) -> None:
            if u == n:
                if len(path) >= 2:
                    ans.append(path[:])
                return
            if nums[u] >= last:
                path.append(nums[u])
                dfs(u + 1, nums[u])
                path.pop()
            if nums[u] != last:
                dfs(u + 1, last)

        dfs(0, -1000)
        return ans


for nums, expected in examples:
    assert sorted(SolutionTakeSkip().findSubsequences(nums)) == sorted(expected)
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp.** Cây có độ sâu `n`, mỗi nút rẽ tối đa 2 nhánh ⇒ `O(2^n)` nút lá, mỗi lá sao chép `O(n)` ⇒ **`O(n · 2^n)`** thời gian; bộ nhớ phụ `O(n)` (đệ quy + path). Lưu ý: cách này **không cắt tỉa mạnh** bằng Cách 3 ở các nhánh "bỏ", vì nó đi hết đến `u == n`; nhưng code ngắn nhất.

### Cách 5 — Xây dần tập dãy theo từng phần tử (không đệ quy)

**Ý tưởng khác hẳn.** Duyệt `nums` từ trái sang phải, giữ `seqs` = **tập mọi dãy không giảm (dài `>= 1`) có thể tạo từ các phần tử đã duyệt**. Khi gặp `x`:

- dãy mới `(x,)` (bắt đầu từ `x`);
- với mỗi dãy `s` trong `seqs` có `s[-1] <= x`, dãy `s + (x,)`.

Gộp chúng vào `seqs` (set tự khử trùng). Cuối cùng lấy các dãy dài `>= 2`.

**Vì sao đúng?** Mọi dãy con không giảm kết thúc tại `x` hoặc là `(x,)`, hoặc là dãy không giảm trước đó (dùng các phần tử đứng trước `x`) cộng thêm `x` với điều kiện số cuối `<= x`. Ta phải dùng bản `seqs` **trước khi** thêm `x` (không dùng `x` hai lần), nên tính `new` trước rồi mới hợp nhất.

In [16]:
def find_build_sets(nums):
    seqs = set()
    for x in nums:
        new = {s + (x,) for s in seqs if s[-1] <= x}
        new.add((x,))
        seqs |= new
    return [list(s) for s in seqs if len(s) >= 2]


for nums, expected in examples:
    got = find_build_sets(nums)
    assert sorted(got) == sorted(expected)
    print(nums, "->", sorted(got))

[4, 6, 7, 7] -> [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
[4, 4, 3, 2, 1] -> [[4, 4]]


In [17]:
class SolutionBuild:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        seqs = set()
        for x in nums:
            new = {s + (x,) for s in seqs if s[-1] <= x}
            new.add((x,))
            seqs |= new
        return [list(s) for s in seqs if len(s) >= 2]


for nums, expected in examples:
    assert sorted(SolutionBuild().findSubsequences(nums)) == sorted(expected)
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp.** Mỗi phần tử duyệt qua toàn bộ `seqs` (tối đa `O(2^n)` dãy), mỗi lần tạo tuple `O(n)` ⇒ **`O(n · 2^n)`** thời gian và **`O(n · 2^n)`** bộ nhớ (lưu hết `seqs`, kể cả dãy dài 1). Nhược điểm: tốn bộ nhớ hơn backtracking, nhưng không có đệ quy và rất dễ chứng minh đúng.

## 8. TRACE từng bước

Trace Cách 3 (backtracking khử trùng theo tầng) với `nums = [4, 6, 7, 7]`. Cột `used` là tập giá trị đã thử ở tầng hiện tại **sau khi** xử lý chỉ số `i`.

| bước | tầng (path hiện tại) | `i` | `nums[i]` | hành động | path sau | ghi nhận |
|---|---|---|---|---|---|---|
| 1 | `[]` | 0 | 4 | lấy 4 | `[4]` | — |
| 2 | `[4]` | 1 | 6 | lấy 6 | `[4,6]` | `[4,6]` |
| 3 | `[4,6]` | 2 | 7 | lấy 7 | `[4,6,7]` | `[4,6,7]` |
| 4 | `[4,6,7]` | 3 | 7 | lấy 7 | `[4,6,7,7]` | `[4,6,7,7]` |
| 5 | `[4,6]` | 3 | 7 | `7 in used` ⇒ bỏ | — | — |
| 6 | `[4]` | 2 | 7 | lấy 7 | `[4,7]` | `[4,7]` |
| 7 | `[4,7]` | 3 | 7 | lấy 7 | `[4,7,7]` | `[4,7,7]` |
| 8 | `[4]` | 3 | 7 | `7 in used` ⇒ bỏ | — | — |
| ... | tiếp tục với tầng gốc `i = 1, 2, 3` | | | | | `[6,7]`, `[6,7,7]`, `[7,7]` |

Hàm `trace_level` in cây đệ quy, thụt lề theo độ sâu.

In [18]:
def trace_level(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(start, depth):
        pad = "  " * depth
        used = set()
        print(f"{pad}dfs(start={start}) path={path}")
        for i in range(start, n):
            x = nums[i]
            if path and x < path[-1]:
                print(f"{pad}  i={i} x={x}: nhỏ hơn số cuối {path[-1]} -> bỏ")
                continue
            if x in used:
                print(f"{pad}  i={i} x={x}: đã dùng làm phần tử kế tiếp ở tầng này -> bỏ (chống trùng)")
                continue
            used.add(x)
            path.append(x)
            note = ""
            if len(path) >= 2:
                ans.append(path[:])
                note = f"  => ghi nhận {path}"
            print(f"{pad}  i={i} x={x}: lấy, path={path}{note}")
            dfs(i + 1, depth + 1)
            path.pop()

    dfs(0, 0)
    print("kết quả:", ans)
    return ans


r = trace_level([4, 6, 7, 7])

dfs(start=0) path=[]
  i=0 x=4: lấy, path=[4]
  dfs(start=1) path=[4]
    i=1 x=6: lấy, path=[4, 6]  => ghi nhận [4, 6]
    dfs(start=2) path=[4, 6]
      i=2 x=7: lấy, path=[4, 6, 7]  => ghi nhận [4, 6, 7]
      dfs(start=3) path=[4, 6, 7]
        i=3 x=7: lấy, path=[4, 6, 7, 7]  => ghi nhận [4, 6, 7, 7]
        dfs(start=4) path=[4, 6, 7, 7]
      i=3 x=7: đã dùng làm phần tử kế tiếp ở tầng này -> bỏ (chống trùng)
    i=2 x=7: lấy, path=[4, 7]  => ghi nhận [4, 7]
    dfs(start=3) path=[4, 7]
      i=3 x=7: lấy, path=[4, 7, 7]  => ghi nhận [4, 7, 7]
      dfs(start=4) path=[4, 7, 7]
    i=3 x=7: đã dùng làm phần tử kế tiếp ở tầng này -> bỏ (chống trùng)
  i=1 x=6: lấy, path=[6]
  dfs(start=2) path=[6]
    i=2 x=7: lấy, path=[6, 7]  => ghi nhận [6, 7]
    dfs(start=3) path=[6, 7]
      i=3 x=7: lấy, path=[6, 7, 7]  => ghi nhận [6, 7, 7]
      dfs(start=4) path=[6, 7, 7]
    i=3 x=7: đã dùng làm phần tử kế tiếp ở tầng này -> bỏ (chống trùng)
  i=2 x=7: lấy, path=[7]
  dfs(start=3) path=

In [19]:
# Trace nhánh "chọn / bỏ" của Cách 4 trên mảng nhỏ có số trùng liên tiếp
def trace_take_skip(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(u, last, depth):
        pad = "  " * depth
        if u == n:
            ok = len(path) >= 2
            print(f"{pad}u={u} (hết) path={path} {'-> ghi nhận' if ok else '-> bỏ (ngắn)'}")
            if ok:
                ans.append(path[:])
            return
        x = nums[u]
        print(f"{pad}u={u} x={x} last={last} path={path}")
        if x >= last:
            print(f"{pad}  lấy {x}")
            path.append(x); dfs(u + 1, x, depth + 1); path.pop()
        if x != last:
            print(f"{pad}  bỏ {x}")
            dfs(u + 1, last, depth + 1)
        else:
            print(f"{pad}  x == last: KHÔNG được bỏ (tránh trùng)")

    dfs(0, -1000, 0)
    print("kết quả:", ans)
    return ans


r = trace_take_skip([1, 1, 2])

u=0 x=1 last=-1000 path=[]
  lấy 1
  u=1 x=1 last=1 path=[1]
    lấy 1
    u=2 x=2 last=1 path=[1, 1]
      lấy 2
      u=3 (hết) path=[1, 1, 2] -> ghi nhận
      bỏ 2
      u=3 (hết) path=[1, 1] -> ghi nhận
    x == last: KHÔNG được bỏ (tránh trùng)
  bỏ 1
  u=1 x=1 last=-1000 path=[]
    lấy 1
    u=2 x=2 last=1 path=[1]
      lấy 2
      u=3 (hết) path=[1, 2] -> ghi nhận
      bỏ 2
      u=3 (hết) path=[1] -> bỏ (ngắn)
    bỏ 1
    u=2 x=2 last=-1000 path=[]
      lấy 2
      u=3 (hết) path=[2] -> bỏ (ngắn)
      bỏ 2
      u=3 (hết) path=[] -> bỏ (ngắn)
kết quả: [[1, 1, 2], [1, 1], [1, 2]]


## 9. Code nộp LeetCode

Ta nộp Cách 3 (khử trùng theo tầng): dễ hiểu, cắt tỉa sớm, mỗi dãy được tạo đúng một lần.

In [20]:
class Solution:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        n = len(nums)
        ans: List[List[int]] = []
        path: List[int] = []

        def dfs(start: int) -> None:
            used = set()  # các giá trị đã thử làm phần tử kế tiếp ở tầng này
            for i in range(start, n):
                if (path and nums[i] < path[-1]) or nums[i] in used:
                    continue
                used.add(nums[i])
                path.append(nums[i])
                if len(path) >= 2:
                    ans.append(path[:])
                dfs(i + 1)
                path.pop()

        dfs(0)
        return ans


for nums, expected in examples:
    got = Solution().findSubsequences(nums)
    assert sorted(got) == sorted(expected)
    print(nums, "->", sorted(got))

[4, 6, 7, 7] -> [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
[4, 4, 3, 2, 1] -> [[4, 4]]


## 10. Vì sao đúng

**Tính đầy đủ.** Mỗi dãy con không giảm `v1 <= v2 <= ... <= vk` (`k >= 2`) có thể dựng bằng cách chọn lần lượt các chỉ số tăng dần. Dùng **lần xuất hiện đầu tiên có thể** của mỗi giá trị: sau khi chọn `v_t` ở chỉ số `p_t`, `v_{t+1}` được chọn ở chỉ số **nhỏ nhất** `> p_t` có giá trị `v_{t+1}`. Đường đi như vậy luôn được duyệt vì ở tầng ứng với tiền tố `v1..vt` ta thử mọi `i` từ `p_t + 1` và `used` chỉ chặn các **lặp lại của giá trị**, không chặn lần đầu tiên.

**Tính không trùng.** Trong cùng một tầng (cùng tiền tố), hai ứng viên có cùng giá trị sẽ sinh ra **cùng tập dãy tiếp theo hoặc tập con của nó** (ứng viên đứng trước có nhiều phần tử còn lại hơn để chọn tiếp). Bỏ ứng viên đứng sau không mất dãy nào. Hai tiền tố khác nhau về giá trị ⇒ dãy khác nhau. Do đó mỗi dãy theo giá trị đúng một lần.

**Tính hợp lệ.** Ta chỉ thêm `nums[i] >= path[-1]` nên dãy luôn không giảm; ta chỉ ghi nhận khi `len(path) >= 2`.

**Cách 4 (chọn / bỏ).** Giả sử số cuối đã chọn là `last` và `nums[u] == last`. Mọi dãy nhận được khi **bỏ** `nums[u]` rồi (có thể) lấy một số `last` ở chỉ số sau đều **cũng** nhận được khi **lấy** `nums[u]` rồi bỏ số sau (dãy giá trị y hệt). Vì vậy bỏ nhánh "bỏ" khi `nums[u] == last` không mất dãy nào, và các dãy còn lại chỉ được sinh bởi đúng một đường.

## 11. Time / Space Complexity

Gọi `n = len(nums)`, `R` là số dãy kết quả (`R <= 2^n`).

- **Thời gian**: `O(n · 2^n)` (trường hợp xấu: mảng tăng chặt, `R ≈ 2^n - n - 1` và mỗi dãy sao chép `O(n)`). Với `n = 15`: tối đa khoảng `32 752` dãy.
- **Bộ nhớ phụ** (ngoài đáp án): `O(n)` cho đường đi + đệ quy (cộng `used` mỗi tầng, tổng `O(n^2)` nhỏ).
- **Bộ nhớ đáp án**: `O(n · R)`.

Vì đầu ra có thể dài `O(n · 2^n)`, **không thể** tốt hơn mức này trong trường hợp xấu — thuật toán là tối ưu theo kích thước output.

## 12. Bộ test

Oracle độc lập: `oracle(nums)` dùng `itertools.combinations` theo chỉ số + kiểm tra định nghĩa + `set` (không dùng bất cứ luật chống trùng nào của các cách). Hàm `norm` sắp xếp để so sánh vì đề cho phép thứ tự bất kỳ.

In [21]:
def oracle(nums):
    res = set()
    for k in range(2, len(nums) + 1):
        for pick in combinations(range(len(nums)), k):
            vals = tuple(nums[i] for i in pick)
            if all(vals[j] <= vals[j + 1] for j in range(k - 1)):
                res.add(vals)
    return sorted(list(v) for v in res)


def norm(lst):
    return sorted(list(x) for x in lst)


def is_valid_answer(nums, ans):
    """Kiểm tra tính hợp lệ: không trùng, mỗi dãy không giảm, dài >= 2, là dãy con của nums."""
    seen = set()
    for seq in ans:
        t = tuple(seq)
        assert t not in seen, ("trùng", seq)
        seen.add(t)
        assert len(seq) >= 2, ("quá ngắn", seq)
        assert all(seq[i] <= seq[i + 1] for i in range(len(seq) - 1)), ("giảm", seq)
        it = iter(nums)
        assert all(any(x == y for y in it) for x in seq), ("không là dãy con", seq)
    return True


SOLUTIONS = [Solution(), SolutionBrute(), SolutionBTSet(), SolutionBTLevel(), SolutionTakeSkip(), SolutionBuild()]
print("oracle([4,6,7,7]) =", oracle([4, 6, 7, 7]))

oracle([4,6,7,7]) = [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]


In [22]:
# 12.1 Ví dụ đề + edge case
cases = [
    ([4, 6, 7, 7], [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]),
    ([4, 4, 3, 2, 1], [[4, 4]]),
    ([1], []),                       # chỉ 1 phần tử
    ([1, 2], [[1, 2]]),
    ([2, 1], []),                    # giảm hoàn toàn
    ([1, 1], [[1, 1]]),
    ([7, 7, 7], [[7, 7], [7, 7, 7]]),
    ([3, 2, 1], []),
    ([1, 2, 3], [[1, 2], [1, 2, 3], [1, 3], [2, 3]]),
    ([-100, 100], [[-100, 100]]),    # biên giá trị
    ([100, -100], []),
    ([1, 3, 2, 2], [[1, 3], [1, 2], [1, 2, 2], [2, 2]]),
]
for nums, expected in cases:
    assert oracle(nums) == norm(expected), (nums, oracle(nums))
    for sol in SOLUTIONS:
        got = sol.findSubsequences(list(nums))
        assert is_valid_answer(nums, got)
        assert norm(got) == norm(expected), (type(sol).__name__, nums, norm(got))
print("OK:", len(cases), "test x", len(SOLUTIONS), "lớp")

OK: 12 test x 6 lớp


In [23]:
# 12.2 Phản ví dụ hay gặp: mảng không được sắp xếp + nhiều số trùng
nums = [1, 2, 1, 2]
print("oracle:", oracle(nums))
for sol in SOLUTIONS:
    assert norm(sol.findSubsequences(nums)) == oracle(nums), type(sol).__name__
# Dãy [1,1,2], [1,2,2], [1,1,2,2]... phụ thuộc thứ tự: [2,1] không bao giờ xuất hiện
assert [2, 1] not in oracle(nums)
print("OK")

oracle: [[1, 1], [1, 1, 2], [1, 2], [1, 2, 2], [2, 2]]
OK


In [24]:
# 12.3 Vét cạn mọi mảng độ dài <= 7 trên bảng chữ cái {1,2,3}
from itertools import product

count = 0
for n in range(1, 8):
    for arr in product((1, 2, 3), repeat=n):
        nums = list(arr)
        exp = oracle(nums)
        for sol in SOLUTIONS:
            assert norm(sol.findSubsequences(nums)) == exp, (type(sol).__name__, nums)
        count += 1
print("OK: đối chiếu", count, "mảng với oracle")

OK: đối chiếu 3279 mảng với oracle


In [25]:
# 12.4 Ngẫu nhiên có seed, đúng ràng buộc: n <= 15, giá trị trong [-100, 100]
random.seed(491)
for _ in range(300):
    n = random.randint(1, 11)
    mode = random.choice(["wide", "narrow", "sorted", "reverse"])
    if mode == "wide":
        nums = [random.randint(-100, 100) for _ in range(n)]
    elif mode == "narrow":
        nums = [random.randint(-2, 2) for _ in range(n)]
    elif mode == "sorted":
        nums = sorted(random.randint(-5, 5) for _ in range(n))
    else:
        nums = sorted((random.randint(-5, 5) for _ in range(n)), reverse=True)
    exp = oracle(nums)
    for sol in SOLUTIONS:
        got = sol.findSubsequences(nums)
        assert is_valid_answer(nums, got)
        assert norm(got) == exp, (type(sol).__name__, nums)
print("OK: 300 mảng ngẫu nhiên n <= 11 (đối chiếu oracle)")

OK: 300 mảng ngẫu nhiên n <= 11 (đối chiếu oracle)


In [26]:
# 12.5 n = 15: đối chiếu giữa các cách nhanh (oracle quá chậm ở đây), kiểm tra số lượng đã biết
fast = [Solution(), SolutionTakeSkip(), SolutionBuild(), SolutionBTSet()]
cases15 = {
    "tăng chặt 1..15": list(range(1, 16)),
    "toàn bằng nhau": [5] * 15,
    "giảm chặt": list(range(15, 0, -1)),
    "xen kẽ": [1, 2] * 7 + [3],
}
for name, nums in cases15.items():
    t0 = time.perf_counter()
    results = [norm(s.findSubsequences(nums)) for s in fast]
    dt = time.perf_counter() - t0
    assert all(r == results[0] for r in results), name
    print(f"{name:<18} -> {len(results[0]):>6} dãy ({dt:.2f}s cho 4 cách)")

assert len(norm(Solution().findSubsequences(list(range(1, 16))))) == 2 ** 15 - 15 - 1   # mọi tập con dài >= 2
assert len(norm(Solution().findSubsequences([5] * 15))) == 14                          # độ dài 2..15
assert norm(Solution().findSubsequences(list(range(15, 0, -1)))) == []
print("OK")

tăng chặt 1..15    ->  32752 dãy (0.24s cho 4 cách)
toàn bằng nhau     ->     14 dãy (0.02s cho 4 cách)
giảm chặt          ->      0 dãy (0.00s cho 4 cách)
xen kẽ             ->     82 dãy (0.00s cho 4 cách)
OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `used` toàn cục thay vì theo từng tầng

Nếu ta lập một `used` duy nhất cho cả bài toán, giá trị `7` bị "tiêu thụ" ở nhánh đầu và các nhánh khác không được dùng nữa ⇒ **mất đáp án**.

In [27]:
def wrong_global_used(nums):
    n = len(nums)
    ans, path = [], []
    used = set()                                     # SAI: toàn cục

    def dfs(start):
        for i in range(start, n):
            if (path and nums[i] < path[-1]) or nums[i] in used:
                continue
            used.add(nums[i])
            path.append(nums[i])
            if len(path) >= 2:
                ans.append(path[:])
            dfs(i + 1)
            path.pop()

    dfs(0)
    return ans


nums = [4, 6, 7, 7]
bad = norm(wrong_global_used(nums))
print("sai:", bad)
print("đúng:", oracle(nums))
print("thiếu:", [x for x in oracle(nums) if x not in bad])

sai: [[4, 6], [4, 6, 7]]
đúng: [[4, 6], [4, 6, 7], [4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]
thiếu: [[4, 6, 7, 7], [4, 7], [4, 7, 7], [6, 7], [6, 7, 7], [7, 7]]


**Vì sao sai?** Sau khi đã thử `7` ở một nhánh, mọi nhánh khác (tiền tố khác) không được dùng `7` nữa. Phải tạo `used = set()` **mới trong mỗi lần gọi** `dfs`. Code đúng: xem `Solution` ở mục 9.

### Lỗi 2: quên sao chép `path` khi ghi nhận

In [28]:
def wrong_no_copy(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(start):
        used = set()
        for i in range(start, n):
            if (path and nums[i] < path[-1]) or nums[i] in used:
                continue
            used.add(nums[i])
            path.append(nums[i])
            if len(path) >= 2:
                ans.append(path)                      # SAI: tham chiếu tới path
            dfs(i + 1)
            path.pop()

    dfs(0)
    return ans


print("sai :", wrong_no_copy([4, 6, 7, 7]))             # toàn list rỗng vì path bị pop hết
print("đúng:", Solution().findSubsequences([4, 6, 7, 7])[:3], "...")

sai : [[], [], [], [], [], [], [], []]
đúng: [[4, 6], [4, 6, 7], [4, 6, 7, 7]] ...


**Vì sao sai?** `ans.append(path)` lưu **cùng một đối tượng** `path`; khi backtrack `path.pop()` làm nó rỗng dần, nên cuối cùng mọi phần tử của `ans` đều là cùng một list (rỗng). Phải lưu `path[:]` hoặc `list(path)`.

### Lỗi 3: sắp xếp `nums` để dễ khử trùng

Nhiều người nhớ mẹo của bài Subsets II ("sắp xếp rồi bỏ qua phần tử bằng phần tử trước") và áp dụng vào đây. Nhưng ở đây **thứ tự gốc quyết định dãy con nào hợp lệ**, sắp xếp sẽ làm xuất hiện dãy không có thật.

In [29]:
nums = [3, 1, 2]
print("oracle (đúng):", oracle(nums))              # chỉ có [1, 2]
print("nếu sắp xếp trước:", oracle(sorted(nums)))  # có thêm [1,3], [2,3], [1,2,3] - SAI

oracle (đúng): [[1, 2]]
nếu sắp xếp trước: [[1, 2], [1, 2, 3], [1, 3], [2, 3]]


**Vì sao sai?** `[1,3]` không phải dãy con của `[3,1,2]` (trong mảng gốc `3` đứng trước `1`). Không sắp xếp; hãy dùng `used` theo tầng (hoặc luật chọn/bỏ).

### Lỗi 4: ghi nhận cả dãy dài 1 hoặc dùng `>` thay vì `>=`

In [30]:
def wrong_len_and_strict(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(start):
        used = set()
        for i in range(start, n):
            if (path and nums[i] <= path[-1]) or nums[i] in used:   # SAI: bỏ dấu bằng
                continue
            used.add(nums[i])
            path.append(nums[i])
            if len(path) >= 1:                                       # SAI: cho cả độ dài 1
                ans.append(path[:])
            dfs(i + 1)
            path.pop()

    dfs(0)
    return ans


print("sai :", norm(wrong_len_and_strict([4, 4, 3, 2, 1])))
print("đúng:", norm(Solution().findSubsequences([4, 4, 3, 2, 1])))

sai : [[1], [2], [3], [4]]
đúng: [[4, 4]]


**Vì sao sai?** (a) Đề yêu cầu **ít nhất 2** phần tử, nên điều kiện ghi nhận là `len(path) >= 2`. (b) "Không giảm" cho phép **bằng nhau**: phải so sánh `nums[i] < path[-1]` để loại, không phải `<=`; nếu loại cả bằng thì mất `[4,4]`.

## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "liệt kê mọi dãy con / tập con thỏa điều kiện" và `n <= 15..20` ⇒ backtracking hoặc bitmask.
- **Chống trùng có hai họ cách:** (1) mảng **đã sắp xếp** ⇒ bỏ qua phần tử bằng phần tử trước (Subsets II, Combination Sum II); (2) mảng **không thể sắp xếp** ⇒ `used` theo từng tầng hoặc luật chọn/bỏ (bài này).
- **Cây đệ quy "for i in range(start, n)"** là khung của combinations; cây "lấy / bỏ" là khung của subsets. Bài này dùng được cả hai.
- **Cắt tỉa sớm** (`nums[i] < path[-1]` ⇒ bỏ) làm cây nhỏ hơn nhiều so với vét cạn.
- **Kiểm tra bằng tập:** viết sẵn oracle bằng `combinations` + `set` rồi so sánh sau `sorted` — rất hiệu quả cho các bài liệt kê.
- Mẹo tự kiểm tra khi viết `used`: hỏi "hai nhánh này có thể sinh ra **cùng một dãy** không?" Nếu có thì phải có cơ chế gộp.

## 15. Biến thể

1. **Chỉ đếm số dãy** (không liệt kê): bằng DP/đếm, có thể nhanh hơn `2^n` khi `n` lớn nhưng việc khử trùng theo giá trị cần cẩn thận (xem bài tập 5).
2. **Dãy tăng chặt** (không cho bằng): đổi `<` thành `<=` ở điều kiện loại; luật Cách 4 đổi thành "mọi `nums[u] <= last` đều bị loại".
3. **Dãy con khác nhau có tổng bằng `k`** hoặc độ dài đúng `k`: thêm tham số `sum` / `len` vào `dfs`.
4. **Subsets II (LeetCode 90)**: tập con khác nhau, nhưng mảng sắp xếp được ⇒ khử trùng bằng "bỏ phần tử bằng phần tử trước ở cùng tầng".
5. **Longest Increasing Subsequence (300)**: chỉ cần độ dài lớn nhất nên giải bằng DP `O(n log n)` thay vì liệt kê.
6. **Distinct Subsequences (115)**: đếm số cách, khác hẳn mục tiêu "dãy khác nhau theo giá trị".

In [31]:
# Biến thể 2: dãy TĂNG CHẶT khác nhau, dài >= 2 (đổi < thành <= ở điều kiện loại)
def strictly_increasing(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(start):
        used = set()
        for i in range(start, n):
            if (path and nums[i] <= path[-1]) or nums[i] in used:
                continue
            used.add(nums[i])
            path.append(nums[i])
            if len(path) >= 2:
                ans.append(path[:])
            dfs(i + 1)
            path.pop()

    dfs(0)
    return ans


print(norm(strictly_increasing([4, 6, 7, 7])))

[[4, 6], [4, 6, 7], [4, 7], [6, 7]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space (phụ) | Khi nào dùng |
|---|---|---|---|---|
| 1. Bitmask vét cạn | thử mọi tập con, lọc rồi khử trùng bằng set | `O(n·2^n)` (luôn đủ `2^n` mask) | `O(n·2^n)` | làm oracle, hiểu bản chất |
| 2. Backtracking + set cuối | cắt nhánh giảm, set gom trùng | `O(n·2^n)` nhưng lặp nhánh trùng | `O(n·2^n)` | khi muốn code đơn giản, `n` nhỏ |
| 3. Backtracking + `used` theo tầng | mỗi dãy sinh đúng 1 lần | `O(n·2^n)` | `O(n)` | **nên nộp**, cắt tỉa tốt |
| 4. Chọn / bỏ có luật chống trùng | 2 nhánh, ép lấy khi bằng số cuối | `O(n·2^n)` | `O(n)` | code ngắn, phỏng vấn |
| 5. Xây dần tập dãy | `seqs` cập nhật theo từng phần tử | `O(n·2^n)` | `O(n·2^n)` | không đệ quy, dễ chứng minh |

In [32]:
# So sánh thời gian thực nghiệm trên mảng "khó": n = 15, nhiều số trùng
random.seed(7)
nums = [random.randint(1, 4) for _ in range(15)]
nums.sort()
print("nums =", nums)
for name, sol in [("bitmask", SolutionBrute()), ("bt+set", SolutionBTSet()),
                  ("bt+used", SolutionBTLevel()), ("chọn/bỏ", SolutionTakeSkip()), ("xây tập", SolutionBuild())]:
    t0 = time.perf_counter()
    r = sol.findSubsequences(nums)
    print(f"{name:<8}: {len(r):>5} dãy, {time.perf_counter() - t0:.3f}s")

nums = [1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4]
bitmask :   379 dãy, 0.053s
bt+set  :   379 dãy, 0.010s
bt+used :   379 dãy, 0.000s
chọn/bỏ :   379 dãy, 0.000s
xây tập :   379 dãy, 0.001s


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `norm(Solution().findSubsequences([1, 1]))`; (b) số dãy của `[2, 2, 2]`; (c) `norm(Solution().findSubsequences([5, 4]))`; (d) với `nums = [1, 2, 2]`, liệt kê kết quả.

**Bài 2 (tính tay).** Với `nums = [1, 3, 2, 3]`, liệt kê **theo giá trị** mọi dãy con không giảm dài `>= 2`, rồi đếm. Gợi ý: nhóm theo phần tử đầu.

**Bài 3 (điền TODO).** Hoàn thành `find_todo(nums)` theo Cách 3.

In [33]:
def find_todo(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(start):
        # TODO 1: used = set()
        # TODO 2: for i in range(start, n): bỏ qua nếu nums[i] nhỏ hơn số cuối của path hoặc nums[i] đã nằm trong used
        # TODO 3: thêm vào used và path; nếu len(path) >= 2 thì ans.append(path[:])
        # TODO 4: dfs(i + 1) rồi path.pop()
        pass

    # TODO 5: gọi dfs(0) và return ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `find_again(nums)` theo Cách 5 (xây dần tập dãy) **không nhìn lại** mục 7. Kiểm tra với `[4,6,7,7]` và `[4,4,3,2,1]`.

**Bài 5 (biến thể / mini challenge).** Viết `count_distinct(nums)`: **chỉ đếm** số dãy không giảm khác nhau dài `>= 2` mà **không liệt kê** (không lưu từng dãy). Gợi ý: đếm theo "dãy kết thúc bằng giá trị `v` ở lần xuất hiện **cuối cùng**" hoặc đếm bằng cây chọn/bỏ ở Cách 4 chỉ cộng biến đếm. Kiểm chứng bằng `len(oracle(nums))`.

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ).** Bài 3: nhớ mỗi lần gọi `dfs` là một **tầng**; `used` sinh ra ở đầu hàm `dfs`, không phải ngoài hàm. Bài 5: Cách 4 chỉ cần thay `ans.append(path[:])` bằng `count += 1`.

**Mức 2 (vừa).** Bài 2: dãy bắt đầu bằng `1`: `[1,3]`, `[1,2]`, `[1,2,3]`, `[1,3,3]`; bắt đầu bằng `3` (chỉ số 1): `[3,3]`; bắt đầu bằng `2`: `[2,3]`. Hãy kiểm tra từng cặp có trùng giá trị không.

**Mức 3 (sát đáp án).** Bài 5: dùng `dfs(u, last)` như Cách 4 nhưng trả về **số** thay vì danh sách: `dfs(u, last)` trả số dãy (kể cả dãy rỗng) từ `u` trở đi; tổng cuối cùng trừ đi các dãy có độ dài 0 và 1 (nhớ đếm độ dài `len(path)` nếu cần). Cách dễ nhất: truyền thêm `length` và chỉ cộng 1 khi `u == n` và `length >= 2`.

## 19. Đáp án tham khảo cho bài tập

In [34]:
# Bài 1
assert norm(Solution().findSubsequences([1, 1])) == [[1, 1]]
assert len(Solution().findSubsequences([2, 2, 2])) == 2                  # [2,2], [2,2,2]
assert Solution().findSubsequences([5, 4]) == []
assert norm(Solution().findSubsequences([1, 2, 2])) == [[1, 2], [1, 2, 2], [2, 2]]

# Bài 2: [1,3,2,3]
assert oracle([1, 3, 2, 3]) == [[1, 2], [1, 2, 3], [1, 3], [1, 3, 3], [2, 3], [3, 3]]
assert len(oracle([1, 3, 2, 3])) == 6
print("bài 1-2 OK")

bài 1-2 OK


In [35]:
# Bài 3
def find_todo_answer(nums):
    n = len(nums)
    ans, path = [], []

    def dfs(start):
        used = set()
        for i in range(start, n):
            if (path and nums[i] < path[-1]) or nums[i] in used:
                continue
            used.add(nums[i])
            path.append(nums[i])
            if len(path) >= 2:
                ans.append(path[:])
            dfs(i + 1)
            path.pop()

    dfs(0)
    return ans


for nums, expected in examples:
    assert norm(find_todo_answer(nums)) == norm(expected)
print("bài 3 OK")

bài 3 OK


In [36]:
# Bài 4
def find_again(nums):
    seqs = set()
    for x in nums:
        new = {s + (x,) for s in seqs if s[-1] <= x}
        new.add((x,))
        seqs |= new
    return [list(s) for s in seqs if len(s) >= 2]


for nums, expected in examples:
    assert norm(find_again(nums)) == norm(expected)
print("bài 4 OK")

bài 4 OK


In [37]:
# Bài 5: chỉ đếm
def count_distinct(nums):
    n = len(nums)

    def dfs(u, last, length):
        if u == n:
            return 1 if length >= 2 else 0
        total = 0
        if nums[u] >= last:
            total += dfs(u + 1, nums[u], length + 1)
        if nums[u] != last:
            total += dfs(u + 1, last, length)
        return total

    return dfs(0, -1000, 0)


for nums, expected in examples:
    assert count_distinct(nums) == len(expected)
random.seed(4910)
for _ in range(300):
    nums = [random.randint(-3, 3) for _ in range(random.randint(1, 10))]
    assert count_distinct(nums) == len(oracle(nums)), nums
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (backtracking, khử trùng theo tầng)

```python
class Solution:
    def findSubsequences(self, nums: List[int]) -> List[List[int]]:
        n = len(nums)
        ans, path = [], []

        def dfs(start):
            used = set()
            for i in range(start, n):
                if (path and nums[i] < path[-1]) or nums[i] in used:
                    continue
                used.add(nums[i])
                path.append(nums[i])
                if len(path) >= 2:
                    ans.append(path[:])
                dfs(i + 1)
                path.pop()

        dfs(0)
        return ans
```

**Checklist**

- [ ] Đọc kỹ: dãy con (giữ thứ tự), **không giảm** (cho phép bằng), **khác nhau**, dài **`>= 2`**?
- [ ] Không sắp xếp `nums`?
- [ ] `used` tạo **trong** mỗi lần gọi `dfs` (theo tầng), không phải toàn cục?
- [ ] Ghi nhận bản sao `path[:]`, không phải `path`?
- [ ] Điều kiện loại là `nums[i] < path[-1]` (không phải `<=`)?
- [ ] Đã thử `[1]`, `[2,1]`, `[7,7,7]`, `[4,4,3,2,1]`, mảng `n = 15` tăng chặt?
- [ ] Đã so sánh với oracle (`combinations` + `set`) trên mảng nhỏ, sau khi `sorted` hai bên?